<a href="https://colab.research.google.com/github/polis-community/red-dwarf/blob/main/docs/notebooks/agora-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
%pip install --quiet red-dwarf[all]@git+https://github.com/polis-community/red-dwarf.git@main

Note: you may need to restart the kernel to use updated packages.


# Agora Pipeline Demo

This notebook demonstrates the **Agora pipeline**, the recommended default for Red Dwarf.

Agora builds on the same core as Polis (PCA + KMeans) but improves statement selection and consensus scoring:

- **Benjamini-Hochberg FDR control** for adaptive, data-driven statement selection (replaces Polis's heuristic `pick_max=5`)
- **Simes' p-value combination** for repness (valid under positive dependence)
- **Effective agreement GAC**: `prod(pa*(1-pd))^(1/n)` penalizes divided groups
- **Zero-vote filtering** to avoid inflating hypothesis counts

## Load Data

We use a local fixture with <100 participants from a real Polis conversation about tech and politics in 2018.

In [10]:
from reddwarf.data_loader import Loader
from reddwarf.utils.statements import process_statements

# You can also load from a Polis URL: Loader(polis_id="r2dfw8eambusb8buvecjt")
loader = Loader(filepaths=[
    "../../tests/fixtures/below-100-ptpts/votes.json",
    "../../tests/fixtures/below-100-ptpts/comments.json",
    "../../tests/fixtures/below-100-ptpts/conversation.json",
])

_, _, mod_out_statement_ids, meta_statement_ids = process_statements(
    statement_data=loader.comments_data
)

# Build a lookup for statement text
stmt_text = {}
for c in loader.comments_data:
    sid = c.get("tid", c.get("statement_id"))
    stmt_text[sid] = c.get("txt", "")

print(f"Loaded {len(loader.votes_data)} votes")
print(f"Statements: {len(loader.comments_data)} total, {len(mod_out_statement_ids)} moderated out")

Loaded 504 votes
Statements: 44 total, 2 moderated out


## Run the Agora Pipeline

In [ ]:
from reddwarf.implementations.agora import run_pipeline

result = run_pipeline(
    votes=loader.votes_data,
    mod_out_statement_ids=mod_out_statement_ids,
    meta_statement_ids=meta_statement_ids,
    random_state=42,
)

# modfication due to change of defnition of run_pipeline in red-dwarf 
result = result.result

n_groups = len(result.ranked_repness)
n_clustered = len(result.participants_df[result.participants_df["to_cluster"]])
n_statements = len(result.statements_df)

print(f"Participants: {n_clustered} clustered into {n_groups} groups")
print(f"Statements: {n_statements - len(mod_out_statement_ids)} active ({len(mod_out_statement_ids)} moderated out)")

Participants: 19 clustered into 3 groups
Statements: 42 active (2 moderated out)


## Representative Statements (Repness)

For each group, statements are ranked by **effect size** (`ra * pa` for agree, `rd * pd` for disagree) and selected using Benjamini-Hochberg FDR control.

The `[*]` marker indicates BH-selected statements.

In [12]:
def truncate(text, max_len=60):
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."

for gid in sorted(result.ranked_repness.keys()):
    print(f"\n--- Group {gid} ---")
    print(f"{'#':>2}  Sel  Dir       Effect  Statement")
    for s in result.ranked_repness[gid]:
        sel = "[*]" if s.selected else "[ ]"
        direction = "agree" if s.repful_for == "agree" else "disagree"
        text = truncate(stmt_text.get(s.statement_id, "?"))
        print(f"{s.rank:>2}  {sel}  {direction:<8}  {s.effect_size:.4f}  s{s.statement_id}: {text}")


--- Group 0 ---
 #  Sel  Dir       Effect  Statement
 1  [*]  disagree  4.8000  s13: Polis statements can be too simplistic and lack context for ...
 2  [*]  agree     4.4800  s1: The prominent display of divisive issues is concerning as po...
 3  [ ]  disagree  3.5556  s29: whether I engage through a text bar would really depend on t...
 4  [ ]  disagree  3.5556  s34: It asks for my email once I’ve got through all the statement...
 5  [*]  disagree  3.4133  s9: Due to the 140 character limit, users tend to give up on com...
 6  [*]  disagree  3.2000  s20: It's hard to communicate technical and quantitative points-o...
 7  [ ]  disagree  2.7500  s27: I wish statements weren't limited to only 140 characters and...
 8  [ ]  disagree  2.6667  s35: Set a slightly higher character limit. Around 165. The curre...
 9  [ ]  agree     1.8000  s22: Polis relies on the participant to put themselves in the oth...
10  [ ]  agree     1.7920  s5: Tooling for multiple moderators is non-existent.
11  

## Cross-Group Comparison

The `group_comment_stats` DataFrame lets you compare how different groups responded to each statement.

In [13]:
group_ids = sorted(result.ranked_repness.keys())
stats = result.group_comment_stats

# Show top repness statement for group 0 across all groups
top_stmt = result.ranked_repness[0][0]
sid = top_stmt.statement_id
print(f"Statement s{sid}: {truncate(stmt_text.get(sid, '?'))}")
print(f"Direction: {top_stmt.repful_for}, Effect size: {top_stmt.effect_size:.4f}")
print()
for g in group_ids:
    row = stats.loc[(g, sid)]
    print(f"  Group {g}: pa={row['pa']:.2f}, pd={row['pd']:.2f} (agree={int(row['na'])}, disagree={int(row['nd'])}, seen={int(row['ns'])})")

Statement s13: Polis statements can be too simplistic and lack context for ...
Direction: disagree, Effect size: 4.8000

  Group 0: pa=0.20, pd=0.80 (agree=0, disagree=3, seen=3)
  Group 1: pa=0.17, pd=0.33 (agree=0, disagree=1, seen=4)
  Group 2: pa=0.91, pd=0.09 (agree=9, disagree=0, seen=9)


## Group-Aware Consensus (GAC)

Agora uses **effective agreement**: `prod(pa * (1-pd))^(1/n_groups)`.

Unlike raw Polis GAC (`prod(pa)^(1/n)`), this penalizes groups that are genuinely divided (high agree AND high disagree).

In [14]:
gac = result.group_aware_consensus

# Top statements by effective agreement (agree direction)
sorted_agree = sorted(
    [(sid, score) for sid, score in gac["agree"].items() if sid not in mod_out_statement_ids],
    key=lambda x: x[1], reverse=True,
)

print("Top 10 statements by group-aware consensus (agree):")
print(f"{'#':>2}  Stmt  GAC     Text")
for rank, (sid, score) in enumerate(sorted_agree[:10], 1):
    text = truncate(stmt_text.get(sid, "?"))
    print(f"{rank:>2}  s{sid:<3}  {score:.4f}  {text}")

Top 10 statements by group-aware consensus (agree):
 #  Stmt  GAC     Text
 1  s26   0.6748  I like how Polis relies on user interaction to analyze comme...
 2  s28   0.6300  Questions that have been 'passed' should be available to ans...
 3  s12   0.5313  Polis is a very simple tool to use for participants.
 4  s30   0.4932  Polis statements are atomic/binary
 5  s38   0.4743  Fix log in
 6  s36   0.4543  Give participants feedback about how their statements are do...
 7  s32   0.4543  There should be a QR code for sharing poll
 8  s18   0.4440  I am worried bad actors can swarm polis with subtly biased s...
 9  s31   0.4293  Nuance in Polis is created by entering multiple statements.
10  s16   0.4269  It sucks that there's no way to undo a vote


## Consensus Statements

Consensus statements use BH FDR control on the one-prop test z-scores, ranked by pa (agree) or pd (disagree).

In [15]:
print("--- Consensus (agree) ---")
print(f"{'#':>2}  Sel  Stmt  Effect  p_adj    Text")
for s in result.ranked_consensus.agree:
    sel = "[*]" if s.selected else "[ ]"
    text = truncate(stmt_text.get(s.statement_id, "?"))
    print(f"{s.rank:>2}  {sel}  s{s.statement_id:<3}  {s.effect_size:.4f}  {s.adjusted_p_value:.4f}  {text}")

print()
print("--- Consensus (disagree) ---")
print(f"{'#':>2}  Sel  Stmt  Effect  p_adj    Text")
for s in result.ranked_consensus.disagree:
    sel = "[*]" if s.selected else "[ ]"
    text = truncate(stmt_text.get(s.statement_id, "?"))
    print(f"{s.rank:>2}  {sel}  s{s.statement_id:<3}  {s.effect_size:.4f}  {s.adjusted_p_value:.4f}  {text}")

--- Consensus (agree) ---
 #  Sel  Stmt  Effect  p_adj    Text
 1  [*]  s26   0.8889  0.0058  I like how Polis relies on user interaction to analyze comme...
 2  [*]  s28   0.8750  0.0083  Questions that have been 'passed' should be available to ans...
 3  [ ]  s32   0.7500  0.2359  There should be a QR code for sharing poll
 4  [ ]  s30   0.7273  0.2359  Polis statements are atomic/binary
 5  [ ]  s18   0.6667  0.2359  I am worried bad actors can swarm polis with subtly biased s...
 6  [ ]  s39   0.6667  0.2359  testing if this works when i submit with keyboard only.
 7  [ ]  s36   0.6667  0.2359  Give participants feedback about how their statements are do...
 8  [ ]  s37   0.6667  0.2359  Admin option of setting a threshold of a successful agreemen...
 9  [ ]  s42   0.6667  0.2359  There is no clear visual hierarchy
10  [ ]  s43   0.6667  0.2359  test
11  [ ]  s41   0.6667  0.2359  the interface is lack of space in between headings
12  [ ]  s40   0.6667  0.2359  Testing out screen r

## Comparison with Polis Selection

For reference, here's what stock Polis selection (`pick_max=5`) produces on the same data.

In [16]:
from reddwarf.utils.stats import select_representative_statements
from reddwarf.utils.consensus import select_consensus_statements

polis_repness = select_representative_statements(
    grouped_stats_df=result.group_comment_stats,
    mod_out_statement_ids=mod_out_statement_ids,
)
polis_consensus = select_consensus_statements(
    vote_matrix=result.raw_vote_matrix,
    mod_out_statement_ids=mod_out_statement_ids,
)

def fmt_stmt(sid, direction):
    d = "a" if direction == "agree" else "d"
    return f"s{sid}({d})"

print("--- Repness comparison ---")
for gid in sorted(result.ranked_repness.keys()):
    polis_stmts = [fmt_stmt(s["tid"], s["repful-for"]) for s in polis_repness.get(gid, [])]
    agora_stmts = [fmt_stmt(s.statement_id, s.repful_for) for s in result.ranked_repness[gid] if s.selected]
    print(f"Group {gid}:")
    print(f"  Polis (pick_max=5): {' '.join(polis_stmts) if polis_stmts else '(none)'}")
    print(f"  Agora (BH fdr=0.10): {' '.join(agora_stmts) if agora_stmts else '(none)'}")

print()
print("--- Consensus comparison ---")
polis_agree = [f"s{s['tid']}" for s in polis_consensus["agree"]]
agora_agree = [f"s{s.statement_id}" for s in result.ranked_consensus.agree if s.selected]
print(f"Agree:")
print(f"  Polis: {' '.join(polis_agree) if polis_agree else '(none)'}")
print(f"  Agora: {' '.join(agora_agree) if agora_agree else '(none)'}")

polis_disagree = [f"s{s['tid']}" for s in polis_consensus["disagree"]]
agora_disagree = [f"s{s.statement_id}" for s in result.ranked_consensus.disagree if s.selected]
print(f"Disagree:")
print(f"  Polis: {' '.join(polis_disagree) if polis_disagree else '(none)'}")
print(f"  Agora: {' '.join(agora_disagree) if agora_disagree else '(none)'}")

--- Repness comparison ---
Group 0:
  Polis (pick_max=5): s1(a) s13(d) s9(d) s20(d) s29(d)
  Agora (BH fdr=0.10): s13(d) s1(a) s9(d) s20(d)
Group 1:
  Polis (pick_max=5): s26(a)
  Agora (BH fdr=0.10): (none)
Group 2:
  Polis (pick_max=5): s13(a) s27(a) s19(a) s9(a) s0(d)
  Agora (BH fdr=0.10): s13(a) s9(a) s27(a) s19(a) s17(a) s20(a) s0(d) s28(a)

--- Consensus comparison ---
Agree:
  Polis: s26 s28 s32 s30 s18
  Agora: s26 s28
Disagree:
  Polis: s0
  Agora: (none)


## Key Differences Summary

| Feature | Polis | Agora |
|---------|-------|-------|
| **Statement selection** | Heuristic `pick_max=5` per group | BH FDR control (adaptive, data-driven) |
| **P-value combination** | z-score thresholds | Simes' method (valid under positive dependence) |
| **Group-aware consensus** | `prod(pa)^(1/n)` (raw agreement) | `prod(pa*(1-pd))^(1/n)` (penalizes divided groups) |
| **Zero-vote handling** | Included in hypothesis testing | Excluded from BH hypothesis count |
| **Output** | Only top-N selected statements | ALL statements ranked with selection flags |

**Use Agora** for production analysis — it provides statistically rigorous, adaptive selection.

**Use Polis** when you need to exactly reproduce upstream Polis behavior for verification or comparison.